# Feature Engineering & Preprocessing

**Цель**: построить полный набор признаков для моделирования продаж.

## Самопроверка перед стартом (Data Preprocessing Rule)

| Вопрос | Ответ |
|--------|-------|
| **Leakage Check** | Темпоральный сплит. Train ≤ 2017-07-31, Val: 2017-08-01—15, Test: 2017-08-16—31. Лаги/rolling строго backward. Энкодеры fit on train. |
| **EDA→Feature** | H1 (землетрясение) → `is_earthquake_period`. H3 (нефть lag=0) → текущая + MA-7. H2 (transferred) → простой `is_holiday`. H5 (школа) → `month` достаточно. |
| **Bias** | 31.3% нулей, BOOKS/BABY CARE ~80%. Решение — на этапе моделирования. |
| **Data Context** | Time series → val = последние 15 дней train (совпадает с test horizon). |
| **Choice Rationale** | Median rolling (выбросы max=124K, P50=11). Forward fill нефти (causal). lag-364 (= 52 недели, совпадение по дню недели). |

## 0. Setup & Data Load

In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

data_dir = os.path.join('..', 'data')
processed_dir = os.path.join(data_dir, 'processed')
artifacts_dir = os.path.join('..', 'artifacts')

train = pd.read_csv(os.path.join(data_dir, 'train.csv'), parse_dates=['date']).drop(columns=['id'])
test = pd.read_csv(os.path.join(data_dir, 'test.csv'), parse_dates=['date']).drop(columns=['id'])
stores = pd.read_csv(os.path.join(data_dir, 'stores.csv'))
oil = pd.read_csv(os.path.join(data_dir, 'oil.csv'), parse_dates=['date'])
holidays = pd.read_csv(os.path.join(data_dir, 'holidays_events.csv'), parse_dates=['date'])

print(f'Train: {train.shape}, date range: {train["date"].min()} — {train["date"].max()}')
print(f'Test:  {test.shape},  date range: {test["date"].min()} — {test["date"].max()}')

Train: (3000888, 5), date range: 2013-01-01 00:00:00 — 2017-08-15 00:00:00
Test:  (28512, 4),  date range: 2017-08-16 00:00:00 — 2017-08-31 00:00:00


## 1. Temporal Split Definition

- **Train**: ≤ 2017-07-31 (для обучения)
- **Val**: 2017-08-01 — 2017-08-15 (валидация, совпадает с горизонтом test)
- **Test**: 2017-08-16 — 2017-08-31 (из `test.csv`, без `sales`)

Конкатенируем train + test в единый DF для расчёта фичей. `sales` в test = NaN → утечки нет.

In [2]:
TRAIN_END = pd.Timestamp('2017-07-31')
VAL_END   = pd.Timestamp('2017-08-15')

test['sales'] = np.nan
df = pd.concat([train, test], ignore_index=True)
df = df.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)

split_labels = np.where(
    df['date'] <= TRAIN_END, 'train',
    np.where(df['date'] <= VAL_END, 'val', 'test')
)
df['split'] = split_labels

for s in ['train', 'val', 'test']:
    n = (df['split'] == s).sum()
    dates = df.loc[df['split'] == s, 'date']
    print(f'{s:>5}: {n:>9,} rows | {dates.min().date()} — {dates.max().date()}')

train: 2,974,158 rows | 2013-01-01 — 2017-07-31
  val:    26,730 rows | 2017-08-01 — 2017-08-15
 test:    28,512 rows | 2017-08-16 — 2017-08-31


## 2. Base Preprocessing

### 2a. Christmas Gaps
EDA выявила 4 пропущенные даты (25 Dec 2013–2016) — магазины закрыты. Вставляем строки с `sales=0` для каждой пары store×family, чтобы лаговые фичи не «перепрыгивали» через дыры.

In [3]:
christmas_dates = pd.to_datetime(['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25'])
existing_dates = df['date'].unique()
missing_xmas = [d for d in christmas_dates if d not in existing_dates]
print(f'Missing Christmas dates: {[str(d.date()) for d in missing_xmas]}')

if missing_xmas:
    store_family = df[['store_nbr', 'family']].drop_duplicates()
    rows = []
    for d in missing_xmas:
        chunk = store_family.copy()
        chunk['date'] = d
        chunk['sales'] = 0.0
        chunk['onpromotion'] = 0
        chunk['split'] = 'train'
        rows.append(chunk)

    xmas_df = pd.concat(rows, ignore_index=True)
    df = pd.concat([df, xmas_df], ignore_index=True)
    df = df.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)
    print(f'Inserted {len(xmas_df)} rows. New shape: {df.shape}')
else:
    print('No missing Christmas dates — nothing to insert.')

Missing Christmas dates: ['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25']


Inserted 7128 rows. New shape: (3036528, 6)


### 2b. Oil Price: Forward Fill

In [4]:
oil = oil.rename(columns={'dcoilwtico': 'oil_price'})
print(f'Oil NaN before ffill: {oil["oil_price"].isna().sum()} / {len(oil)} ({oil["oil_price"].isna().mean():.1%})')

oil = oil.sort_values('date')
oil['oil_price'] = oil['oil_price'].ffill().bfill()
print(f'Oil NaN after ffill:  {oil["oil_price"].isna().sum()}')

Oil NaN before ffill: 43 / 1218 (3.5%)
Oil NaN after ffill:  0


## 3. Merge External Data

In [5]:
pre_merge = df.shape[0]

df = df.merge(stores[['store_nbr', 'city', 'state', 'type', 'cluster']], on='store_nbr', how='left')
df = df.merge(oil[['date', 'oil_price']], on='date', how='left')

df['oil_price'] = df.groupby(['store_nbr', 'family'])['oil_price'].transform(lambda s: s.ffill().bfill())

assert df.shape[0] == pre_merge, f'Row count changed after merge: {pre_merge} -> {df.shape[0]}'
print(f'Shape after merges: {df.shape}')
print(f'Oil NaN remaining: {df["oil_price"].isna().sum()}')
print(f'Columns: {list(df.columns)}')

Shape after merges: (3036528, 11)
Oil NaN remaining: 0
Columns: ['date', 'store_nbr', 'family', 'sales', 'onpromotion', 'split', 'city', 'state', 'type', 'cluster', 'oil_price']


## 4. Feature Engineering

### 4a. Date Features [HIGH]

In [6]:
df['day_of_week']  = df['date'].dt.dayofweek
df['day_of_month'] = df['date'].dt.day
df['month']        = df['date'].dt.month
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
df['quarter']      = df['date'].dt.quarter
df['is_weekend']   = (df['day_of_week'] >= 5).astype(np.int8)
df['year']         = df['date'].dt.year

print('Date features added:', ['day_of_week', 'day_of_month', 'month', 'week_of_year', 'quarter', 'is_weekend', 'year'])

Date features added: ['day_of_week', 'day_of_month', 'month', 'week_of_year', 'quarter', 'is_weekend', 'year']


### 4b. Lag Features [HIGH]

Лаги по `sales` для каждой группы (store_nbr, family).
- lag-7, lag-14, lag-28: недельная/двухнедельная/месячная сезонность.
- lag-364: годовая сезонность (52 недели × 7 = совпадение дня недели).

In [7]:
LAG_DAYS = [7, 14, 28, 364]

grouped = df.groupby(['store_nbr', 'family'])['sales']
for lag in LAG_DAYS:
    df[f'lag_{lag}'] = grouped.shift(lag)
    print(f'  lag_{lag}: NaN = {df[f"lag_{lag}"].isna().sum():,}')

print(f'\nLag features added: {[f"lag_{d}" for d in LAG_DAYS]}')

  lag_7: NaN = 28,512


  lag_14: NaN = 28,512
  lag_28: NaN = 49,896
  lag_364: NaN = 648,648

Lag features added: ['lag_7', 'lag_14', 'lag_28', 'lag_364']


### 4c. Rolling Aggregates [HIGH]

Mean по окнам 7, 14, 30 + median по окну 7 (устойчив к выбросам max=124K при P50=11).
`shift(1)` перед rolling — чтобы не утечь текущий день. `min_periods=1` — не теряем начало рядов.

In [8]:
ROLLING_WINDOWS = [7, 14, 30]

df['_shifted_sales'] = df.groupby(['store_nbr', 'family'])['sales'].shift(1)

# C-optimized groupby().rolling() — без Python lambda overhead
grp = df.groupby(['store_nbr', 'family'])['_shifted_sales']

for w in ROLLING_WINDOWS:
    rolled = grp.rolling(w, min_periods=1).mean()
    df[f'rolling_mean_{w}'] = rolled.droplevel([0, 1]).sort_index()

rolled_med = grp.rolling(7, min_periods=1).median()
df['rolling_median_7'] = rolled_med.droplevel([0, 1]).sort_index()

df = df.drop(columns=['_shifted_sales'])

rolling_cols = [f'rolling_mean_{w}' for w in ROLLING_WINDOWS] + ['rolling_median_7']
print(f'Rolling features added: {rolling_cols}')
print(f'NaN in rolling_mean_7: {df["rolling_mean_7"].isna().sum():,}')

Rolling features added: ['rolling_mean_7', 'rolling_mean_14', 'rolling_mean_30', 'rolling_median_7']
NaN in rolling_mean_7: 17,820


### 4d. Oil MA-7 [HIGH]

Текущая цена уже в `oil_price`. Добавляем скользящее среднее за 7 дней (по дате, глобально).

In [9]:
oil_daily = df[['date', 'oil_price']].drop_duplicates('date').sort_values('date').copy()
oil_daily['oil_ma_7'] = oil_daily['oil_price'].rolling(7, min_periods=1).mean()

df = df.merge(oil_daily[['date', 'oil_ma_7']], on='date', how='left')
print(f'oil_ma_7 NaN: {df["oil_ma_7"].isna().sum()}')

oil_ma_7 NaN: 0


### 4e. Holiday Features [MED]

H2 опровергла пользу сложной логики `transferred`. Делаем просто:
- `is_holiday` — бинарный флаг (National holidays + Regional matching by state + Local matching by city).
- `days_to_next_holiday` / `days_since_last_holiday` — дистанция до ближайшего праздника.

In [10]:
national = holidays[holidays['locale'] == 'National'][['date']].drop_duplicates()
national['is_national_holiday'] = 1

regional = holidays[holidays['locale'] == 'Regional'][['date', 'locale_name']].drop_duplicates()
regional = regional.rename(columns={'locale_name': 'state'})
regional['is_regional_holiday'] = 1

local = holidays[holidays['locale'] == 'Local'][['date', 'locale_name']].drop_duplicates()
local = local.rename(columns={'locale_name': 'city'})
local['is_local_holiday'] = 1

df = df.merge(national, on='date', how='left')
df = df.merge(regional, on=['date', 'state'], how='left')
df = df.merge(local, on=['date', 'city'], how='left')

df['is_holiday'] = (
    df[['is_national_holiday', 'is_regional_holiday', 'is_local_holiday']]
    .fillna(0).max(axis=1).astype(np.int8)
)
df = df.drop(columns=['is_national_holiday', 'is_regional_holiday', 'is_local_holiday'])

print(f'is_holiday distribution:\n{df["is_holiday"].value_counts()}')

# --- Days to/since nearest holiday (vectorized) ---
all_holiday_dates = np.sort(holidays['date'].unique())
hd = all_holiday_dates.astype('datetime64[D]')
dates = df['date'].values.astype('datetime64[D]')

idx = np.searchsorted(hd, dates)

# days to next holiday
idx_next = np.clip(idx, 0, len(hd) - 1)
diff_next = (hd[idx_next] - dates).astype(int)
mask_past = diff_next < 0
idx_next2 = np.clip(idx + 1, 0, len(hd) - 1)
diff_next[mask_past] = (hd[idx_next2[mask_past]] - dates[mask_past]).astype(int)
df['days_to_next_holiday'] = np.clip(diff_next, 0, 999)

# days since last holiday
idx_prev = np.clip(idx - 1, 0, len(hd) - 1)
diff_prev = (dates - hd[idx_prev]).astype(int)
diff_prev[diff_prev < 0] = 999
df['days_since_last_holiday'] = diff_prev

print(f'\ndays_to_next_holiday — mean: {df["days_to_next_holiday"].mean():.1f}, max: {df["days_to_next_holiday"].max()}')
print(f'days_since_last_holiday — mean: {df["days_since_last_holiday"].mean():.1f}, max: {df["days_since_last_holiday"].max()}')

is_holiday distribution:
is_holiday
0    2761737
1     274791
Name: count, dtype: int64

days_to_next_holiday — mean: 9.7, max: 59
days_since_last_holiday — mean: 10.6, max: 60


### 4f. Earthquake Flag [MED]

H1 подтверждена: lift ~142-147% по всей стране. Глобальный бинарный флаг на 30 дней.

In [11]:
EQ_START = pd.Timestamp('2016-04-16')
EQ_END   = pd.Timestamp('2016-05-15')

df['is_earthquake_period'] = ((df['date'] >= EQ_START) & (df['date'] <= EQ_END)).astype(np.int8)
print(f'Earthquake rows: {df["is_earthquake_period"].sum():,}')

Earthquake rows: 53,460


### 4g. Promotion Features [MED]

`onpromotion` as-is + rolling среднее промо за 7 дней (per store×family).

In [12]:
rolled_promo = (
    df.groupby(['store_nbr', 'family'])['onpromotion']
    .rolling(7, min_periods=1).mean()
)
df['promo_rolling_mean_7'] = rolled_promo.droplevel([0, 1]).sort_index()

print(f'promo_rolling_mean_7 NaN: {df["promo_rolling_mean_7"].isna().sum()}')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

promo_rolling_mean_7 NaN: 0
Memory usage: 0.75 GB


### 4h. Payday Flag [LOW]

Статистически не подтвердилось (H4, p=0.43), но дешёвая фича — оставляем.

In [13]:
last_day_of_month = df['date'].dt.is_month_end
is_15th = df['day_of_month'] == 15

df['is_payday'] = (last_day_of_month | is_15th).astype(np.int8)
print(f'Payday rows: {df["is_payday"].sum():,} ({df["is_payday"].mean():.1%})')

Payday rows: 199,584 (6.6%)


## 5. Encoding

`LabelEncoder` fit **строго на train split** → transform на всех. Tree-модели (LightGBM/XGBoost) не нуждаются в scaling.

In [14]:
CAT_COLS = ['family', 'city', 'state', 'type']
train_mask = df['split'] == 'train'
encoders = {}

for col in CAT_COLS:
    le = LabelEncoder()
    le.fit(df.loc[train_mask, col].astype(str))
    mapping = {cls: idx for idx, cls in enumerate(le.classes_)}
    df[col] = df[col].astype(str).map(mapping).fillna(-1).astype(int)
    encoders[col] = le
    n_unseen = (df[col] == -1).sum()
    print(f'  {col}: {len(le.classes_)} classes, unseen in val/test: {n_unseen}')

joblib.dump(encoders, os.path.join(artifacts_dir, 'label_encoders.pkl'))
print(f'\nEncoders saved to {artifacts_dir}/label_encoders.pkl')

  family: 33 classes, unseen in val/test: 0


  city: 22 classes, unseen in val/test: 0


  state: 16 classes, unseen in val/test: 0


  type: 5 classes, unseen in val/test: 0

Encoders saved to ..\artifacts/label_encoders.pkl


## 6. Validation & Leakage Check

In [15]:
print('=== Shape per split ===')
for s in ['train', 'val', 'test']:
    subset = df[df['split'] == s]
    print(f'  {s:>5}: {subset.shape[0]:>10,} rows')

print('\n=== NaN per feature (total) ===')
nan_counts = df.isna().sum()
nan_cols = nan_counts[nan_counts > 0]
if len(nan_cols) > 0:
    for col, cnt in nan_cols.items():
        print(f'  {col}: {cnt:,} ({cnt / len(df):.1%})')
else:
    print('  No NaN!')

print('\n=== Leakage sanity: val/test should NOT have non-NaN sales from future ===')
val_sales_nan = df.loc[df['split'] == 'val', 'sales'].isna().sum()
test_sales_nan = df.loc[df['split'] == 'test', 'sales'].isna().sum()
test_total = (df['split'] == 'test').sum()
print(f'  Val sales NaN: {val_sales_nan} (expected: 0 — val has real sales for evaluation)')
print(f'  Test sales NaN: {test_sales_nan} / {test_total} (expected: all NaN)')

print('\n=== Lag sanity: correlation lag_7 vs sales on train ===')
train_valid = df[(df['split'] == 'train') & df['lag_7'].notna() & df['sales'].notna()]
corr = train_valid['lag_7'].corr(train_valid['sales'])
print(f'  Pearson r(lag_7, sales) on train: {corr:.4f}')

=== Shape per split ===


  train:  2,981,286 rows
    val:     26,730 rows
   test:     28,512 rows

=== NaN per feature (total) ===


  sales: 28,512 (0.9%)
  lag_7: 28,512 (0.9%)
  lag_14: 28,512 (0.9%)
  lag_28: 49,896 (1.6%)
  lag_364: 648,648 (21.4%)
  rolling_mean_7: 17,820 (0.6%)
  rolling_mean_14: 5,346 (0.2%)
  rolling_mean_30: 1,782 (0.1%)
  rolling_median_7: 17,820 (0.6%)

=== Leakage sanity: val/test should NOT have non-NaN sales from future ===
  Val sales NaN: 0 (expected: 0 — val has real sales for evaluation)
  Test sales NaN: 28512 / 28512 (expected: all NaN)

=== Lag sanity: correlation lag_7 vs sales on train ===


  Pearson r(lag_7, sales) on train: 0.9362


## 7. Save Processed Data

In [16]:
for split_name in ['train', 'val', 'test']:
    mask = df['split'] == split_name
    subset = df.loc[mask].drop(columns=['split'])
    path = os.path.join(processed_dir, f'{split_name}_fe.parquet')
    subset.to_parquet(path, index=False)
    print(f'  {split_name:>5}: {subset.shape[0]:>10,} rows x {subset.shape[1]} cols -> {split_name}_fe.parquet')
    del subset

print(f'\nAll columns ({len(df.columns) - 1}):')
print([c for c in df.columns if c != 'split'])

  train:  2,981,286 rows x 32 cols -> train_fe.parquet
    val:     26,730 rows x 32 cols -> val_fe.parquet


   test:     28,512 rows x 32 cols -> test_fe.parquet

All columns (32):
['date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city', 'state', 'type', 'cluster', 'oil_price', 'day_of_week', 'day_of_month', 'month', 'week_of_year', 'quarter', 'is_weekend', 'year', 'lag_7', 'lag_14', 'lag_28', 'lag_364', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_30', 'rolling_median_7', 'oil_ma_7', 'is_holiday', 'days_to_next_holiday', 'days_since_last_holiday', 'is_earthquake_period', 'promo_rolling_mean_7', 'is_payday']
